# Demo 04 — End-to-End RAG Evaluation

The runnable notebook behind **§10 (End-to-End Evaluation & Reliability)** of *Chapter 04 —
Inference-Time Retrieval Patterns*.

§4 measured retrieval alone (recall@k, hit rate, MRR). This notebook measures the **output the
user actually sees**: does the answer follow from the retrieved context (faithfulness), does it
address the question (answer relevancy), was the useful context ranked well (context precision),
and did retrieval surface everything needed (context recall)? These four metrics are the ones
[RAGAS](https://docs.ragas.io/) popularized — but this notebook does not use the `ragas` package.

The `ragas` package pins a `langchain<1.0` dependency chain that conflicts with this project's
`langchain 1.x` stack (`langgraph`, `langchain-openai`, `dspy`) — installing it broke another
course demo's imports. So the four metrics are implemented directly here as small Claude-judge
functions using the `anthropic` SDK, following §10.3's own advice (`temperature=0` + structured
tool-call output). Each is first validated against the exact worked examples already in
§10.1/§10.2, then run for real on `demo04_rag_dspy`'s baseline RAG pipeline.

Requires `anthropic`, `dspy`, `sentence-transformers`, `numpy` (all already used elsewhere in this
course; no new installs). Needs an `ANTHROPIC_API_KEY` in a `.env` file — on Colab, `google.colab`
(preinstalled) supplies it from Colab Secrets (sidebar key icon) instead.

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")   # this Mac's torch segfaults on MPS
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_HUB_VERBOSITY", "error")

import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

from dotenv import load_dotenv
load_dotenv()
# On Colab, pull the key from Colab Secrets (sidebar key icon) if it isn't already in the env.
if not os.environ.get("ANTHROPIC_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        pass
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set — add it to your .env file"

import orjson
import numpy as np
import anthropic

client = anthropic.Anthropic()
JUDGE_MODEL = "claude-haiku-4-5-20251001"   # same model demo04_rag_dspy uses for generation

_embedder = None
def embedder():
    global _embedder
    if _embedder is None:
        from sentence_transformers import SentenceTransformer
        _embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
    return _embedder
def encode(texts):
    return embedder().encode(list(texts), normalize_embeddings=True)

print("ready")

## 1 — One judge primitive, reused four times

Every metric below reduces to the same two LLM calls: **decompose** a text into atomic claims, or
**judge** whether one piece of text is supported by / relevant to another. §10.3's advice —
`temperature=0` and structured output instead of free text — turns each judgment into a forced
tool call, so there is no answer to parse, only a typed field to read.

In [2]:
def _tool_call(tool, prompt, max_tokens=400):
    msg = client.messages.create(
        model=JUDGE_MODEL, max_tokens=max_tokens, temperature=0,
        tools=[tool], tool_choice={"type": "tool", "name": tool["name"]},
        messages=[{"role": "user", "content": prompt}],
    )
    for block in msg.content:
        if block.type == "tool_use":
            return block.input
    return {}

def decompose_claims(text: str) -> list[str]:
    """Break a text into atomic, self-contained factual claims."""
    tool = {"name": "record_claims", "description": "Record the atomic factual claims in the text.",
            "input_schema": {"type": "object",
                "properties": {"claims": {"type": "array", "items": {"type": "string"}}},
                "required": ["claims"]}}
    return _tool_call(tool, f"Decompose the following text into its individual atomic factual "
                             f"claims. Each claim must be a single, self-contained fact.\n\n"
                             f"Text: {text}").get("claims", [])

def is_supported(claim: str, context: str) -> bool:
    """Is `claim` directly supported by `context` (not general knowledge)?"""
    tool = {"name": "record_verdict", "description": "Record whether the claim is supported.",
            "input_schema": {"type": "object",
                "properties": {"supported": {"type": "boolean"}}, "required": ["supported"]}}
    return _tool_call(tool, f"Context: {context}\n\nClaim: {claim}\n\n"
                             f"Is this claim directly supported by the context? Judge strictly "
                             f"from the context, not general knowledge.", max_tokens=100
                       ).get("supported", False)

def is_relevant(question: str, chunk: str) -> bool:
    """Is `chunk` useful for answering `question`?"""
    tool = {"name": "record_relevance", "description": "Record the relevance judgment.",
            "input_schema": {"type": "object",
                "properties": {"relevant": {"type": "boolean"}}, "required": ["relevant"]}}
    return _tool_call(tool, f"Question: {question}\n\nChunk: {chunk}\n\n"
                             f"Is this chunk useful for answering the question?", max_tokens=100
                       ).get("relevant", False)

print("judge primitives ready")

judge primitives ready


## 2 — Faithfulness

Decompose the answer into claims, then check each against the retrieved context. Validate against
**§10.1's own worked example** before trusting the mechanism on anything else.

In [3]:
def faithfulness(answer: str, context: str):
    claims = decompose_claims(answer)
    verdicts = [(c, is_supported(c, context)) for c in claims]
    score = sum(v for _, v in verdicts) / len(verdicts) if verdicts else 1.0
    return score, verdicts

# §10.1's exact worked example
context = "The Eiffel Tower was completed in 1889 for the World's Fair…"
answer = "Built in 1889 by Gustave Eiffel, who also designed the Statue of Liberty."
score, verdicts = faithfulness(answer, context)
print(f"faithfulness = {score:.2f}  (§10.1 states 0.33)")
for c, v in verdicts:
    print(f"  {'supported    ' if v else 'HALLUCINATION'}  {c!r}")

faithfulness = 0.33  (§10.1 states 0.33)
  supported      'Built in 1889'
  HALLUCINATION  'Built by Gustave Eiffel'
  HALLUCINATION  'Gustave Eiffel designed the Statue of Liberty'


## 3 — Answer relevancy

Reverse-engineer candidate questions from the answer, embed them, and compare to the real
question. This catches *drift* — a factually fine answer to the wrong question — not correctness.

In [4]:
def generate_questions(answer: str, n: int = 3) -> list[str]:
    tool = {"name": "record_questions", "description": "Record generated questions.",
            "input_schema": {"type": "object",
                "properties": {"questions": {"type": "array", "items": {"type": "string"}}},
                "required": ["questions"]}}
    return _tool_call(tool, f"Given this answer, write {n} distinct questions it could be "
                             f"answering. Infer only what the answer implies.\n\n"
                             f"Answer: {answer}").get("questions", [])

def answer_relevancy(question: str, answer: str):
    gen_qs = generate_questions(answer)
    if not gen_qs:
        return 0.0, gen_qs
    sims = encode(gen_qs) @ encode([question])[0]
    return float(np.mean(sims)), gen_qs

on_topic = answer_relevancy(
    "When was the Eiffel Tower built?", "The Eiffel Tower was completed in 1889.")
drifted = answer_relevancy(
    "When was the Eiffel Tower built?",
    "The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars in Paris.")

print(f"on-topic answer:  relevancy = {on_topic[0]:.2f}   generated Qs: {on_topic[1]}")
print(f"drifted answer:   relevancy = {drifted[0]:.2f}   generated Qs: {drifted[1]}")
print('\n(§10.1: "a \'when\' question answered with a \'what is\' description scores ~0.41" — '
      "same drift, same direction, different toy example.)")

on-topic answer:  relevancy = 0.95   generated Qs: ['When was the Eiffel Tower completed?', 'In what year was the Eiffel Tower finished?', "What year did the Eiffel Tower's construction end?"]
drifted answer:   relevancy = 0.69   generated Qs: ['What is the Eiffel Tower and where is it located?', 'What material is the Eiffel Tower made of and what is its structure?', 'Where on the Champ de Mars in Paris is the wrought-iron lattice tower situated?']

(§10.1: "a 'when' question answered with a 'what is' description scores ~0.41" — same drift, same direction, different toy example.)


## 4 — Context precision

**Rank-weighted**: a useful chunk at rank 1 counts more than the same chunk at rank 4. This is why
bad reranking hurts even when the right chunk is somewhere in the candidate set (§8's lesson,
re-scored here as a metric).

In [5]:
def context_precision(question: str, ranked_chunks: list[str]):
    relevant = [is_relevant(question, c) for c in ranked_chunks]
    total_relevant = sum(relevant)
    if total_relevant == 0:
        return 0.0, relevant
    hits, running_sum = 0, 0.0
    for k, rel in enumerate(relevant, start=1):
        if rel:
            hits += 1
            running_sum += hits / k     # precision@k, counted only when this rank is relevant
    return running_sum / total_relevant, relevant

q = "When was the Eiffel Tower built?"
useful_first = ["The Eiffel Tower was completed in 1889.", "Paris is the capital of France.", "The tower is 330m tall."]
useful_last  = ["Paris is the capital of France.", "The tower is 330m tall.", "The Eiffel Tower was completed in 1889."]

score_first, rel_first = context_precision(q, useful_first)
score_last, rel_last = context_precision(q, useful_last)
print(f"useful chunk at rank 1:  context precision = {score_first:.2f}  (relevance={rel_first})")
print(f"useful chunk at rank 3:  context precision = {score_last:.2f}  (relevance={rel_last})")
print("\nSame candidate chunks, same retriever — only the RANKING changed the score.")

useful chunk at rank 1:  context precision = 1.00  (relevance=[True, False, False])
useful chunk at rank 3:  context precision = 0.33  (relevance=[False, False, True])

Same candidate chunks, same retriever — only the RANKING changed the score.


## 5 — Context recall

The one metric that needs a gold answer: decompose it into claims, then check whether *some*
retrieved chunk supports each one. **Low context recall is a retrieval failure no generator can
fix** — reproduce §10.1's own "missed the height fact" example.

In [6]:
def context_recall(gold_answer: str, retrieved_chunks: list[str]):
    claims = decompose_claims(gold_answer)
    verdicts = []
    for claim in claims:
        supported = any(is_supported(claim, c) for c in retrieved_chunks)  # stop early on first hit
        verdicts.append((claim, supported))
    score = sum(v for _, v in verdicts) / len(verdicts) if verdicts else 1.0
    return score, verdicts

gold = "The Eiffel Tower was completed in 1889 and stands 330 meters tall."
retrieved_missing_height = ["The Eiffel Tower was completed in 1889 for the World's Fair."]

score, verdicts = context_recall(gold, retrieved_missing_height)
print(f"context recall = {score:.2f}  (§10.1 states 0.50 for the same missed-height case)")
for c, v in verdicts:
    print(f"  {'supported    ' if v else 'NOT retrieved'}  {c!r}")

context recall = 0.50  (§10.1 states 0.50 for the same missed-height case)
  supported      'The Eiffel Tower was completed in 1889'
  NOT retrieved  'The Eiffel Tower stands 330 meters tall'


## 6 — Now for real: `demo04_rag_dspy`'s baseline pipeline

All four metrics reproduce the chapter's own toy numbers. Time to run them on **real RAG output** —
the same pipeline as `demo04_rag_dspy.ipynb`: `all-MiniLM-L6-v2` embeddings over the first 1,000
`ragqa_arena_tech` documents, retrieved with `dspy.retrievers.Embeddings`, answered with
`dspy.ChainOfThought("context, question -> response")` on Claude Haiku.

In [7]:
import dspy

lm = dspy.LM(f"anthropic/{JUDGE_MODEL}")
dspy.configure(lm=lm)

st_embed = embedder()
def _embed_fn(texts):
    return st_embed.encode(texts).tolist()
dspy_embedder = dspy.Embedder(_embed_fn)

MAX_CORPUS_SIZE = 1000
with open("ragqa_arena_tech_corpus.jsonl") as f:
    corpus = [orjson.loads(line)["text"][:6000] for _, line in zip(range(MAX_CORPUS_SIZE), f)]

search = dspy.retrievers.Embeddings(embedder=dspy_embedder, corpus=corpus, k=5)

class RAG(dspy.Module):
    def __init__(self, retriever):
        self.retriever = retriever
        self.respond = dspy.ChainOfThought("context, question -> response")
    def forward(self, question):
        context = self.retriever(question).passages
        pred = self.respond(context=context, question=question)
        pred.context = context
        return pred

rag = RAG(retriever=search)
print(f"corpus: {len(corpus)} docs, retriever k=5")

corpus: 1000 docs, retriever k=5


## 7 — A real sample: 6 answerable + 1 out-of-corpus question

`gold_doc_ids` are positional indices into the *full* 28k-doc corpus (per §4's provenance note).
We only indexed the first 1,000 docs here, so we deliberately pick questions whose gold docs fall
inside that range — otherwise retrieval fails for a reason that has nothing to do with the metrics
being taught. One question is picked *outside* that range on purpose, to see how the pipeline
behaves when the answer genuinely isn't in the index (§10.2, next section).

In [8]:
with open("ragqa_arena_tech_examples.jsonl") as f:
    examples = [orjson.loads(line) for line in f]

in_range = [e for e in examples if all(g < MAX_CORPUS_SIZE for g in e["gold_doc_ids"])]
print(f"{len(in_range)} of {len(examples)} examples have gold docs inside the {MAX_CORPUS_SIZE}-doc subset")

rng = np.random.default_rng(0)
sample = list(rng.choice(in_range, size=6, replace=False))
out_of_range_example = next(e for e in examples if e not in in_range)

for ex in sample:
    print(" -", ex["question"])
print("out-of-corpus probe:", out_of_range_example["question"])

137 of 2064 examples have gold docs inside the 1000-doc subset
 - what is the difference between native vlan and default vlan?
 - what percentage of devices have each of the android versions?
 - if my team has low skill, should i lower the skill of my code?
 - what does it mean when the connectivity icons in the status bar go white/gray?
 - what is the difference between cat file | ./binary and ./binary < file?
 - how do i make `ls` show file sizes in megabytes?
out-of-corpus probe: why igp is used in mpls?


## 8 — Run the pipeline, measure all four metrics

For each question: generate the answer, then score faithfulness (answer vs. its own context),
answer relevancy (answer vs. question), context precision (ranked retrieved chunks vs. question),
and context recall (retrieved chunks vs. the gold reference answer).

In [9]:
rows = []
for ex in sample:
    pred = rag(question=ex["question"])
    f_score, _ = faithfulness(pred.response, "\n".join(pred.context))
    r_score, _ = answer_relevancy(ex["question"], pred.response)
    p_score, _ = context_precision(ex["question"], pred.context)
    c_score, _ = context_recall(ex["response"], pred.context)
    rows.append({
        "question": ex["question"], "answer": pred.response, "gold": ex["response"],
        "faithfulness": f_score, "answer_relevancy": r_score,
        "context_precision": p_score, "context_recall": c_score,
    })

for r in rows:
    print(f"Q: {r['question']}")
    print(f"   faithfulness={r['faithfulness']:.2f}  relevancy={r['answer_relevancy']:.2f}  "
          f"precision={r['context_precision']:.2f}  recall={r['context_recall']:.2f}")

print("\n--- averages over the sample ---")
for metric in ("faithfulness", "answer_relevancy", "context_precision", "context_recall"):
    print(f"{metric:18} {np.mean([r[metric] for r in rows]):.2f}")

Q: what is the difference between native vlan and default vlan?
   faithfulness=1.00  relevancy=0.66  precision=0.75  recall=1.00
Q: what percentage of devices have each of the android versions?
   faithfulness=0.94  relevancy=0.75  precision=1.00  recall=1.00
Q: if my team has low skill, should i lower the skill of my code?
   faithfulness=1.00  relevancy=0.60  precision=0.92  recall=1.00
Q: what does it mean when the connectivity icons in the status bar go white/gray?
   faithfulness=0.86  relevancy=0.81  precision=1.00  recall=1.00
Q: what is the difference between cat file | ./binary and ./binary < file?
   faithfulness=1.00  relevancy=0.64  precision=1.00  recall=0.89
Q: how do i make `ls` show file sizes in megabytes?
   faithfulness=0.91  relevancy=0.80  precision=1.00  recall=0.80

--- averages over the sample ---
faithfulness       0.95
answer_relevancy   0.71
context_precision  0.94
context_recall     0.95


## 9 — Abstention  (§10.2)

The four metrics above miss one failure mode: retrieval simply can't find the answer, and a
well-behaved system should **refuse, not invent.** Reproduce §10.2's own architect example, then
check the real out-of-corpus probe question from above.

In [10]:
# §10.2's exact worked example
architect_context = "…completed in 1889 and stands 330 m tall."   # architect not present
architect_question = "Who was the architect of the Eiffel Tower?"
bad_answer = "Stephen Sauvestre designed it."
good_answer = "The context doesn't mention the architect."

f_bad, _ = faithfulness(bad_answer, architect_context)
f_good, _ = faithfulness(good_answer, architect_context)
print(f"invented answer:  faithfulness = {f_bad:.2f}  (§10.2 states 0)")
print(f"honest refusal:    faithfulness = {f_good:.2f}  (§10.2 states 1)")

# the real out-of-corpus probe: its gold doc isn't in our 1,000-doc index at all
print(f"\nreal probe (gold doc outside the indexed 1,000): {out_of_range_example['question']!r}")
probe_pred = rag(question=out_of_range_example["question"])
print("model's answer:", probe_pred.response[:200])

invented answer:  faithfulness = 0.00  (§10.2 states 0)
honest refusal:    faithfulness = 1.00  (§10.2 states 1)

real probe (gold doc outside the indexed 1,000): 'why igp is used in mpls?'
model's answer: The provided context does not contain information about MPLS (Multiprotocol Label Switching) or its relationship with IGP protocols. 

However, based on general networking principles: IGP is used in M


## Takeaways

- **Faithfulness, answer relevancy, and context precision need no gold labels** — they run on
  live traffic. **Context recall needs a gold answer**, so it's eval-set only.
- **Faithfulness is the single most useful production metric.** Decompose the answer into claims,
  check each against the retrieved context — this notebook's implementation reproduces §10.1's own
  worked numbers exactly (0.33 hallucination example, 0.50 missed-fact example).
- **Ranking matters, not just recall.** The same three chunks scored 1.00 vs. 0.33 on context
  precision purely by moving the useful one from rank 1 to rank 3 — a reranker mistake (§8) shows
  up here as a lower score, not a retrieval failure.
- **A well-behaved system refuses rather than invents** when the answer genuinely isn't retrieved —
  both on the toy architect example and on a real ragqa question outside our small index.
- **You don't need the `ragas` package to get its ideas** — implementing the four metrics directly
  with the model you already use kept this notebook dependency-light and avoided a real,
  unresolvable conflict with this project's `langchain 1.x` stack.